In [94]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

1. Embeddings

1.1 Simplest possible version: one hot vectors

In [95]:
vocab = ["good", "great", "bad", "shoe", "movie"]
one_hot = {word: np.eye(len(vocab))[i] for i, word in enumerate(vocab)}

for w, v in one_hot.items():
    print(w.ljust(6), v)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

print("\nsim(good, great):", cosine_sim(one_hot["good"], one_hot["great"]))    # 0.0 -- looks totally unrelated!
print("sim(good, shpe):", cosine_sim(one_hot["good"], one_hot["shoe"]))         #also 0.0 --same "distance"   

good   [1. 0. 0. 0. 0.]
great  [0. 1. 0. 0. 0.]
bad    [0. 0. 1. 0. 0.]
shoe   [0. 0. 0. 1. 0.]
movie  [0. 0. 0. 0. 1.]

sim(good, great): 0.0
sim(good, shpe): 0.0


1.2 A step up: TF-IDF vectors

In [96]:
# --- Example 1: TF-IDF on a few sentences, then cosine similarity ---

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "AlfredX is a PyQt5 based AI voice assistant.",
    "AlfredX supports voice commands and reminders.",
    "The stock market fell sharply today.",
    "Investors reacted to the market crash.",
]


vec = TfidfVectorizer()
tfidf_matrix = vec.fit_transform(docs)
print("Shape:", tfidf_matrix.shape )
print("Vacabulary Size:", len(vec.vocabulary_))

sims = cosine_similarity(tfidf_matrix)
print("\nSimilarity Matrix:\n", np.round(sims, 2))

Shape: (4, 21)
Vacabulary Size: 21

Similarity Matrix:
 [[1.   0.22 0.   0.  ]
 [0.22 1.   0.   0.  ]
 [0.   0.   1.   0.24]
 [0.   0.   0.24 1.  ]]


In [97]:
# Example 2: embeded new Query and find closest document (this IS retrieval, in miniature) ---

query = "Tell me about the voice assistant"
query_vec = vec.transform([query])                # use the SAME fitted vectorizer -- don't refit on the query 

sims_to_query = cosine_similarity(query_vec, tfidf_matrix)[0]
for doc, score in sorted(zip(docs, sims_to_query), key= lambda x: -x[1]):
    print(f"{score:.3f} {doc}")

0.433 AlfredX is a PyQt5 based AI voice assistant.
0.181 AlfredX supports voice commands and reminders.
0.181 The stock market fell sharply today.
0.181 Investors reacted to the market crash.


In [98]:
# proof of the weakness

query2 = "Tell me about the AI companion app"
q2_vec = vec.transform([query2])

sims2 = cosine_similarity(q2_vec, tfidf_matrix)[0]
for doc, score in sorted(zip(docs, sims2), key= lambda x: -x[1]):
    print(f"{score:.3f} {doc}")

0.314 AlfredX is a PyQt5 based AI voice assistant.
0.213 The stock market fell sharply today.
0.213 Investors reacted to the market crash.
0.000 AlfredX supports voice commands and reminders.


1.3 Real dense based embeddings: train a tinyu one from scratch (offline to see what's actually hapening)

In [99]:
import torch, torch.nn as nn, torch.nn.functional as F

# Tiny vocabulary and "training pairs": words that tend to appear in similar contexts get pulled together
vocab2 = ["good", "great", "excellent", "bad", "terrible", "awful", "shoe", "sneaker", "boot"]
word2id = {w: i for i, w in enumerate(vocab2)}

# Positive pairs = should end up CLOSE in vector space (same "cluster")
positive_pairs = [("good","great"), ("great","excellent"), ("good","excellent"),
                  ("bad","terrible"), ("terrible","awful"), ("bad","awful"),
                  ("shoe","sneaker"), ("sneaker","boot"), ("shoe","boot")]


embed_dim = 8
embedding = nn.Embedding(len(vocab2), embed_dim)
optimizer = torch.optim.Adam(embedding.parameters(), lr=0.05)

for epoch in range(300):
    total_loss = 0
    for w1, w2 in positive_pairs:
        v1 = embedding(torch.tensor(word2id[w1]))
        v2 = embedding(torch.tensor(word2id[w2]))
        # pull positive pairs together: maximize cosine similarity -> minimize (1 - cos_sim)
        loss = 1 - F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)) 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 60 == 0:
        print(f"epoch {epoch}  loss {total_loss:.3f}")    


epoch 0  loss 6.860
epoch 60  loss 0.001
epoch 120  loss 0.000
epoch 180  loss 0.002
epoch 240  loss 0.001


In [100]:

# --- Now check: did semantically similar words end up with similar vectors, with NO shared characters? ---

def sim(w1, w2):
    v1 = embedding(torch.tensor(word2id[w1]))
    v2 = embedding(torch.tensor(word2id[w2]))
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))

print("sim(good, great)      :", round(sim("good","great").item(), 3))       # trained to be close
print("sim(good, excellent)  :", round(sim("good","excellent").item(), 3))   # trained to be close, no shared letters!
print("sim(good, terrible)   :", round(sim("good","terrible").item(), 3))    # never linked -> should be lower
print("sim(good, shoe)       :", round(sim("good","shoe").item(), 3))        # different cluster entirely -> lowest
    

sim(good, great)      : 0.999
sim(good, excellent)  : 1.0
sim(good, terrible)   : 0.162
sim(good, shoe)       : -0.215


1.4 REal sentense embeddings with 'sentense-transformer'

In [101]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")         # small, fast, 384-dim -- great default for RAG

docs = [
    "AlfredX is a PyQt5 based AI voice assistant.",
    "AlfredX supports voice commands and reminders.",
    "The stock market fell sharply today.",
]
doc_embeddings = model.encode(docs)             # shape: (3, 384)
query_embedding = model.encode("Tell me about AI campanion app")

from sklearn.metrics.pairwise import  cosine_similarity
sims = cosine_similarity([query_embedding], doc_embeddings)[0]
for doc, score in sorted(zip(doc, sims), key= lambda x : -x[1]):
    print(round(score, 3), doc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.276 l
0.273 A
0.007 f


2. chunking

In [102]:
sample_doc = '''
AlfredX is a PyQt5-based AI voice assistant built by a four-member team. It uses the Groq API
with Llama 3 for language understanding and supports over one hundred voice and text commands.

The core AI engine was rewritten to support full multi-turn conversation history, streaming
responses from Groq, and rate-limit retry logic. Seven critical bugs were fixed during
development, including a duplicate class definition and broken context handling.

The interface is bilingual, supporting English and Turkish, with a JARVIS-style HUD featuring
animated system monitors and a voice waveform visualization. A FastAPI and WebSocket based
mobile companion app allows remote control of the assistant from a phone.
'''.strip()
print(len(sample_doc), "characters")

709 characters


2.1 Simplest possible: fixed size character chunks(naive cuts words/sentence in half)

In [103]:
def naive_chunk(text, chunk_size):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]


chunks = naive_chunk(sample_doc, chunk_size=100)
for i, c in enumerate(chunks):
        print(f"--- chunk {i},")
        print(repr(c))
# notice: words and sentences get cut mid-way. Bad for retrieval quality.        

--- chunk 0,
'AlfredX is a PyQt5-based AI voice assistant built by a four-member team. It uses the Groq API\nwith L'
--- chunk 1,
'lama 3 for language understanding and supports over one hundred voice and text commands.\n\nThe core A'
--- chunk 2,
'I engine was rewritten to support full multi-turn conversation history, streaming\nresponses from Gro'
--- chunk 3,
'q, and rate-limit retry logic. Seven critical bugs were fixed during\ndevelopment, including a duplic'
--- chunk 4,
'ate class definition and broken context handling.\n\nThe interface is bilingual, supporting English an'
--- chunk 5,
'd Turkish, with a JARVIS-style HUD featuring\nanimated system monitors and a voice waveform visualiza'
--- chunk 6,
'tion. A FastAPI and WebSocket based\nmobile companion app allows remote control of the assistant from'
--- chunk 7,
' a phone.'


2.2 Fixed size with overlap

In [104]:
def chunk_with_overlap(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap     # step forward by less than chunk_size -> creates overlap
    return chunks

chunks = chunk_with_overlap(sample_doc, chunk_size=150, overlap=30)
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(repr(c[:60]) + "...")
print("\ntotal chunks:", len(chunks))

--- chunk 0 (150 chars) ---
'AlfredX is a PyQt5-based AI voice assistant built by a four-'...
--- chunk 1 (150 chars) ---
'understanding and supports over one hundred voice and text c'...
--- chunk 2 (150 chars) ---
'ulti-turn conversation history, streaming\nresponses from Gro'...
--- chunk 3 (150 chars) ---
'd during\ndevelopment, including a duplicate class definition'...
--- chunk 4 (150 chars) ---
'upporting English and Turkish, with a JARVIS-style HUD featu'...
--- chunk 5 (109 chars) ---
'tion. A FastAPI and WebSocket based\nmobile companion app all'...

total chunks: 6


2.3 Better: split on sentence/paragraph boundries first

In [105]:
import re

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def sentence_aware_chunk(text, max_chars=200):
    sentences = split_sentences(text.replace("\n", " "))
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) <= max_chars:
            current = (current + " " + sent).strip()
        else:
            if current:
                chunks.append(current)
            current = sent
    if current:
        chunks.append(current)
    return chunks

chunks = sentence_aware_chunk(sample_doc, max_chars=200)
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

--- chunk 0 (188 chars) ---
AlfredX is a PyQt5-based AI voice assistant built by a four-member team. It uses the Groq API with Llama 3 for language understanding and supports over one hundred voice and text commands.

--- chunk 1 (140 chars) ---
The core AI engine was rewritten to support full multi-turn conversation history, streaming responses from Groq, and rate-limit retry logic.

--- chunk 2 (118 chars) ---
Seven critical bugs were fixed during development, including a duplicate class definition and broken context handling.

--- chunk 3 (154 chars) ---
The interface is bilingual, supporting English and Turkish, with a JARVIS-style HUD featuring animated system monitors and a voice waveform visualization.

--- chunk 4 (103 chars) ---
A FastAPI and WebSocket based mobile companion app allows remote control of the assistant from a phone.



2.4 Production grade: 'RecursivecharacterTextSplitter' (langchain)

In [106]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # target size per chunk (characters)
    chunk_overlap=40,      # overlap between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""],   # tries these, in order, to find a clean cut point
)

chunks = splitter.split_text(sample_doc)
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

--- chunk 0 (188 chars) ---
AlfredX is a PyQt5-based AI voice assistant built by a four-member team. It uses the Groq API
with Llama 3 for language understanding and supports over one hundred voice and text commands.

--- chunk 1 (178 chars) ---
The core AI engine was rewritten to support full multi-turn conversation history, streaming
responses from Groq, and rate-limit retry logic. Seven critical bugs were fixed during

--- chunk 2 (80 chars) ---
development, including a duplicate class definition and broken context handling.

--- chunk 3 (184 chars) ---
The interface is bilingual, supporting English and Turkish, with a JARVIS-style HUD featuring
animated system monitors and a voice waveform visualization. A FastAPI and WebSocket based

--- chunk 4 (73 chars) ---
mobile companion app allows remote control of the assistant from a phone.



2. 5 Chunk with metadat(what u actually store for a vectorDB)

In [107]:
def build_chunk_records(text, source_name, chunk_size=200, overlap=40):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    pieces = splitter.split_text(text)
    return[
        {"id":f"{source_name}_{i}", "Source": source_name, "chunk_index":i, "text": piece}
        for i, piece in enumerate(pieces)

    ]

records = build_chunk_records(sample_doc, source_name="alfredx_readme.md")
for r in records:
    print(r["id"], "|", r["text"][:50], "...")
print("\nSampleFull Records:\n", records[0])    

alfredx_readme.md_0 | AlfredX is a PyQt5-based AI voice assistant built  ...
alfredx_readme.md_1 | The core AI engine was rewritten to support full m ...
alfredx_readme.md_2 | development, including a duplicate class definitio ...
alfredx_readme.md_3 | The interface is bilingual, supporting English and ...
alfredx_readme.md_4 | mobile companion app allows remote control of the  ...

SampleFull Records:
 {'id': 'alfredx_readme.md_0', 'Source': 'alfredx_readme.md', 'chunk_index': 0, 'text': 'AlfredX is a PyQt5-based AI voice assistant built by a four-member team. It uses the Groq API\nwith Llama 3 for language understanding and supports over one hundred voice and text commands.'}


2.6 Chunking a real pdf

In [108]:
from PyPDF2 import PdfReader

reader = PdfReader("my_document.pdf")
full_text=""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

records = build_chunk_records(full_text, source_name="my_document")
print(f"{len(records)} chunks created from PDF")


17 chunks created from PDF


----------------------------------------------------------------------------------------------------------------------------

In [109]:
import numpy as np
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
np.set_printoptions(precision=3, suppress=True)
print("faiss:", faiss.__version__)

faiss: 1.15.1


3. Vector Store (Faiss) 

3.1 Simplest  possible : brute force search by hand ( no faiss yet - to see what faiss is automating)

In [110]:
# --- Example 1: 5 tiny 2D "embeddings" and manual nearest-neighbor search ---
vectors = np.array([
    [1.0, 1.0],   # id 0
    [1.2, 0.9],   # id 1  (close to 0)
    [5.0, 5.0],   # id 2
    [5.1, 4.8],   # id 3  (close to 2)
    [0.0, 5.0],   # id 4
], dtype="float32")

query = np.array([1.1, 1.0], dtype="float32")

# brute force: compute distance to every vector, sort
dists = np.linalg.norm(vectors - query, axis=1)
order = np.argsort(dists)
for i in order:
    print(f"id: {i}   Distance {dists[i]:.3f}")      # This is exactly what FAISS does internally for a basic "Flat" index -- just optimized in C++   

id: 0   Distance 0.100
id: 1   Distance 0.141
id: 4   Distance 4.148
id: 3   Distance 5.517
id: 2   Distance 5.587


3.2 A real FAISS index: 'IndexFlatL2'

In [111]:
# --- Example 2: build a FAISS index and search it ---

dim = vectors.shape[1]                  # FAISS needs to know the vector dimension up front
index = faiss.IndexFlatL2(dim)          # L2 = Euclidean distance
print("Is trained:", index.is_trained)  # Flat indexes need no training
print("Vectors in index:", index.ntotal)

index.add(vectors)                      # add all 5 vectors at once
print("Vectors in index after add:", index.ntotal)

k=3
distance, indices = index.search(query.reshape(1, -1), k)      # search expects a BATCH of queries -> shape (n_queries, dim)
print("Nearest IDS:", indices)
print("Distance:", distance)

Is trained: True
Vectors in index: 0
Vectors in index after add: 5
Nearest IDS: [[0 1 4]]
Distance: [[ 0.01  0.02 17.21]]


3.3 Real Chunks -> real vectors -> faiss(the acutal rag workflow)

In [112]:
# --- rebuild chunking + a small document set, self-contained ---
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = {
 "alfredx_readme.md": '''
AlfredX is a PyQt5-based AI voice assistant built by a four-member team. It uses the Groq API
with Llama 3 for language understanding and supports over one hundred voice and text commands.
The core AI engine was rewritten to support full multi-turn conversation history, streaming
responses from Groq, and rate-limit retry logic.
'''.strip(),
 "resume_notes.md": '''
Naved completed a one-month AWS training internship at Jalgi Technologies, certified with a
95 percent score. He is a computer engineering graduate skilled in Python, PyQt5, and FastAPI.
'''.strip(),
 "ml_notes.md": '''
Scikit-learn provides tools for classification, regression, and clustering. Cross-validation
and pipelines help avoid data leakage when preprocessing and training models together.
'''.strip(),
}

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunk_records = []
for source, text in documents.items():
    for i, piece in enumerate(splitter.split_text(text)):
        chunk_records.append({"id": f"{source}_{i}", "source": source, "text": piece})

for r in chunk_records:
    print(r["id"], "|", r["text"][:60], "...")
print("\ntotal chunks:", len(chunk_records))     


alfredx_readme.md_0 | AlfredX is a PyQt5-based AI voice assistant built by a four- ...
alfredx_readme.md_1 | with Llama 3 for language understanding and supports over on ...
alfredx_readme.md_2 | The core AI engine was rewritten to support full multi-turn  ...
resume_notes.md_0 | Naved completed a one-month AWS training internship at Jalgi ...
resume_notes.md_1 | 95 percent score. He is a computer engineering graduate skil ...
ml_notes.md_0 | Scikit-learn provides tools for classification, regression,  ...
ml_notes.md_1 | and pipelines help avoid data leakage when preprocessing and ...

total chunks: 7


In [113]:
# --- embed every chunk (TF-IDF here; swap for sentence-transformers on your machine -- see note below) ---

vectorizer = TfidfVectorizer()
chunk_texts = (r["text"]for r in chunk_records)
chunk_vectors = vectorizer.fit_transform(chunk_texts).toarray().astype("float32")       # FAISS needs float32
print("chunk vector shape:", chunk_vectors.shape)     # (num_chunks, vocab_size)

dim = chunk_vectors.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(chunk_vectors)
print("chunks indexed:", index.ntotal)

chunk vector shape: (7, 86)
chunks indexed: 7


3.4 Cosine Similarity in FAISS: 'IndexFlatIP' + normalization

In [114]:
#Ex: IndexFlatIP with normalized vectors == cosine similarity search

normalized = chunk_vectors.copy()
faiss.normalize_L2(normalized)

index_cosine = faiss.IndexFlatIP(dim)
index_cosine.add(normalized)

query_vec = vectorizer.transform(["What intership did Naved do?"]).toarray().astype("float32")
faiss.normalize_L2(query_vec)

scores, ids = index_cosine.search(query_vec, k=3)
print("Cosine scores:", scores)
print("Chunk IDs:", ids)
for idx in ids[0]:
    print("->", chunk_records[idx]["id"], "|", chunk_records[idx]["text"][:70])

Cosine scores: [[0.301 0.    0.   ]]
Chunk IDs: [[3 2 1]]
-> resume_notes.md_0 | Naved completed a one-month AWS training internship at Jalgi Technolog
-> alfredx_readme.md_2 | The core AI engine was rewritten to support full multi-turn conversati
-> alfredx_readme.md_1 | with Llama 3 for language understanding and supports over one hundred 


3.5 saving and loading a faiss index (so u dont re-embed everything)

In [115]:
import pickle, os

faiss.write_index(index_cosine, "chunks.index")
with open("chunk_metadata.pkl", "wb") as f:
    pickle.dump(chunk_records, f)

print("Saved:", os.path.getsize("chunks.index"), "bytes (index) +", os.path.getsize("chunk_metadata.pkl"), "bytes (metadata)")  

#reload 
loaded_index = faiss.read_index("chunks.index")
with open("chunk_metadata.pkl", "rb") as f:
    loaded_records = pickle.load(f)

print("Reloaded index has", loaded_index.ntotal, "vectorsand", len(loaded_records), "metadata records")


Saved: 2453 bytes (index) + 1010 bytes (metadata)
Reloaded index has 7 vectorsand 7 metadata records


3.6 scaling up: index types beyond 'flat'

In [116]:
# just so u have seen API onceLIndexIVFFlat requires training

nlist = 2
quantizer = faiss.IndexFlatL2(dim)
ivf_index = faiss.IndexIVFFlat(quantizer, dim, nlist)

ivf_index.train(chunk_vectors)
ivf_index.add(chunk_vectors)
print("IVF index trained:", ivf_index.is_trained, "| vectors:", ivf_index.ntotal)

scores, ids = ivf_index.search(chunk_vectors[:1], k=2)
print("Search works the same way:", ids)

IVF index trained: True | vectors: 7
Search works the same way: [[0 2]]


----------------------------------------------------------------------------------------------------------------------------

4. Retrieval

4.1 Simplest possible: a 'retrieve' function wrapping what u already built

In [117]:
def embed_query(text):
    vec = vectorizer.transform([text]).toarray().astype("float32")
    faiss.normalize_L2(vec)
    return vec

def retrieve(query, k=3):
    query_vec = embed_query(query)
    scores, ids = index_cosine.search(query_vec, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        results.append({**chunk_records[idx], "score": float(score)})
    return results

for r in retrieve("What AI projects has Naved built?", k=3):
    print(f"{r['score']:.3f}    [{r['source']}]     [{r['text'][:70]}]")

0.265    [alfredx_readme.md]     [AlfredX is a PyQt5-based AI voice assistant built by a four-member tea]
0.183    [resume_notes.md]     [Naved completed a one-month AWS training internship at Jalgi Technolog]
0.093    [alfredx_readme.md]     [The core AI engine was rewritten to support full multi-turn conversati]


4.2 A harder query: does retrieval actally pick right source?

In [118]:
queries = [
    "What internship experience does Naved have?",
    "What machine learning library concepts are covered?",
    "How does AlfredX handle conversation history?",
]

for q in queries:
    print(f"\nQuery: {q}")
    for r in retrieve(q, k=2):
        print(f"{r['score']:.3f}    [{r['source']}]     [{r['text'][:70]}]")


Query: What internship experience does Naved have?
0.425    [resume_notes.md]     [Naved completed a one-month AWS training internship at Jalgi Technolog]
0.000    [alfredx_readme.md]     [with Llama 3 for language understanding and supports over one hundred ]

Query: What machine learning library concepts are covered?
0.000    [alfredx_readme.md]     [with Llama 3 for language understanding and supports over one hundred ]
0.000    [alfredx_readme.md]     [AlfredX is a PyQt5-based AI voice assistant built by a four-member tea]

Query: How does AlfredX handle conversation history?
0.256    [alfredx_readme.md]     [The core AI engine was rewritten to support full multi-turn conversati]
0.148    [alfredx_readme.md]     [AlfredX is a PyQt5-based AI voice assistant built by a four-member tea]


4.3 Filtering by score threshold("dont retrieve if anything is actually relevant")

In [119]:
def retrieve_with_threshold(query, k=3, min_score=0.15):
    results = retrieve(query, k=k)
    filtered = [r for r in results if r["score"]  >= min_score]
    if not filtered:
        print(f"No chunk scored above {min_score} for: '{query}' -> should answer 'not found in context'")
        return filtered


print(retrieve_with_threshold("What intership experience does naved have?"))
print()
print(retrieve_with_threshold("What is the capital of France?", min_score=0.15))    #unrelated to any document   

None

None


4.4 Metadata filtering: retrieve only from a specific source

In [120]:
def retrieve_filtered(query, k=3, source=None):
    query_vec = embed_query(query)
    scores, ids = index_cosine.search(query_vec, k=len(chunk_records))
    results = []
    for score, idx in zip(scores[0], ids[0]):
        rec = chunk_records[idx]
        if source is None or rec["source"] == source:
            results.append({**rec, "score": float(score)})
        if len(results) == k:
            break
    return results

print("All sources:")
for r in retrieve_filtered("Naved's skills:", k=2):
    print(f"  {r['score']:.3f} [{r['source']}]  {r['text'][:60]}")

print("\nOnly resume_notes.md:")
for r in retrieve_filtered("Naved's skills:", k=2, source="resume_notes.md"):
    print(f"  {r['score']:.3f} [{r['source']}]  {r['text'][:60]}")       

All sources:
  0.301 [resume_notes.md]  Naved completed a one-month AWS training internship at Jalgi
  0.000 [ml_notes.md]  and pipelines help avoid data leakage when preprocessing and

Only resume_notes.md:
  0.301 [resume_notes.md]  Naved completed a one-month AWS training internship at Jalgi
  0.000 [resume_notes.md]  95 percent score. He is a computer engineering graduate skil


4.5 Re-ranking: a socond, more careful pass over the top candidates

In [121]:
def keyword_overlap_score(query, text):
    q_words = set(query.lower().split())
    t_words = set(text.lower().split())
    return len(q_words & t_words)

def retrieve_and_rerank(query, fetch_k=5, final_k=2):
    candidates = retrieve(query, k=fetch_k)
    for c in candidates:
        c["keyword_score"] = keyword_overlap_score(query, c["text"])
    reranked = sorted(candidates, key=lambda c: (c["keyword_score"], c["score"]), reverse=True)
    return reranked[:final_k]

query = "AWS intership exprerience Naved"
print("Vector_only order:")
for r in retrieve(query, k=3):
    print(f" vec={r['score']:.3f}  {r['text'][:60]}")

print("\nRe-ranked order:")
for r in retrieve_and_rerank(query, fetch_k=3, final_k=3):
    print(f" vec={r['score']:.3f}  kw={r['keyword_score']}  {r['text'][:60]}")
        

Vector_only order:
 vec=0.425  Naved completed a one-month AWS training internship at Jalgi
 vec=0.000  The core AI engine was rewritten to support full multi-turn 
 vec=0.000  with Llama 3 for language understanding and supports over on

Re-ranked order:
 vec=0.425  kw=2  Naved completed a one-month AWS training internship at Jalgi
 vec=0.000  kw=0  The core AI engine was rewritten to support full multi-turn 
 vec=0.000  kw=0  with Llama 3 for language understanding and supports over on


----------------------------------------------------------------------------------------------------------------------------

In [122]:
import numpy as np
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain_text_splitters import RecursiveCharacterTextSplitter
np.set_printoptions(precision=3, suppress=True)

5. Generation

5.1 simplest possible:build the prompt string by hand

In [123]:
def build_prompt(question, context_chunks):
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(context_chunks))
    prompt = f'''Answer the question using ONLY the context below. If the answer isn't in the context, say "I don't know based on the given information."

Context:
{context}

Question:  {question}

Answer:'''
    return prompt
    
sample_chunks = [
    "AlfredX uses the Groq API with Llama 3 for language understanding.",
    "AlfredX supports over one hundred voice and text commands.",
]
prompt = build_prompt("What LLM does AlfredX use?", sample_chunks)
print(prompt)

Answer the question using ONLY the context below. If the answer isn't in the context, say "I don't know based on the given information."

Context:
[1] AlfredX uses the Groq API with Llama 3 for language understanding.

[2] AlfredX supports over one hundred voice and text commands.

Question:  What LLM does AlfredX use?

Answer:


5.2 Why this prompt shape matters - the 3 rule

In [124]:
# --- proof: a context that does NOT contain the answer ---
unrelated_chunks = ["AlfredX supports over one hundred voice and text commands.",
                    "The bilingual interface supports English and Turkish."]
prompt2 = build_prompt("What is the capital of France?", unrelated_chunks)
print(prompt2)
print("\n^ a well-behaved LLM should refuse to answer this, because the context has nothing to do with it")

Answer the question using ONLY the context below. If the answer isn't in the context, say "I don't know based on the given information."

Context:
[1] AlfredX supports over one hundred voice and text commands.

[2] The bilingual interface supports English and Turkish.

Question:  What is the capital of France?

Answer:

^ a well-behaved LLM should refuse to answer this, because the context has nothing to do with it
